In [1]:
# %% Imports & Setup
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from torch.optim import lr_scheduler
from time import time
from einops import rearrange
import h5py

# For reproducibility (if needed)
np.random.seed(42)
torch.manual_seed(42)


In [2]:
# %% Preprocessing Functions
def ExtractPatches(img, GT=None, windowSize=3):
    """
    Divide the input image into patches for training.
    If GT is None, a dummy ground truth is created.
    """
    if GT is None:
        GT = np.ones_like(img[0, :, :])
    margin = int((windowSize - 1) / 2)
    img = np.pad(img, pad_width=((0, 0), (margin, margin), (margin, margin)), mode='edge')
    GT = np.pad(GT, pad_width=((margin, margin), (margin, margin)))
    pos_in_image = np.asarray(np.where(GT != 0)).T
    labels = GT[pos_in_image[:, 0], pos_in_image[:, 1]]
    patches = []
    for i in pos_in_image:
        patch = img[:, i[0]-margin:i[0]+margin+1, i[1]-margin:i[1]+margin+1]
        patches.append(patch)
    return np.asanyarray(patches), labels, pos_in_image

def ScaleData(X, min_value, max_value):
    """
    Scale the data X to the range [0, 1].
    """
    X = X.astype(np.float16)
    X -= np.float16(min_value)
    X /= (np.float16(max_value) - np.float16(min_value))
    return X

def extract_overlapping_patches(image, patch_size, stride):
    """
    Extract overlapping patches from an image.
    
    Args:
        image (numpy.ndarray): Input image array.
        patch_size (int): Size of each patch (height and width).
        stride (int): Stride for extracting patches.
    
    Returns:
        numpy.ndarray: Extracted patches.
    """
    # If the image has a channel dimension
    if image.ndim == 3:
        patches = []
        # Slide a window over the height and width dimensions
        for i in range(0, image.shape[1] - patch_size + 1, stride):
            for j in range(0, image.shape[2] - patch_size + 1, stride):
                patch = image[:, i:i+patch_size, j:j+patch_size]
                patches.append(patch)
        return np.array(patches)
    else:
        patches = []
        for i in range(0, image.shape[0] - patch_size + 1, stride):
            for j in range(0, image.shape[1] - patch_size + 1, stride):
                patch = image[i:i+patch_size, j:j+patch_size]
                patches.append(patch)
        return np.array(patches)


In [3]:
# %% Data Loading and Preprocessing with .mat Files

import os
import h5py
import numpy as np

def ScaleData(X, min_value, max_value):
    """
    Scale the data X between 0 and 1.
    """
    X = X.astype(np.float32)
    X -= np.float32(min_value)
    X /= (np.float32(max_value) - np.float32(min_value))
    return X

def load_hdf5_mat_file(file_path):
    """
    Load a .mat file and extract hyperspectral image and ground truth data.
    
    Args:
        file_path (str): Path to the .mat file.
    
    Returns:
        np.ndarray, np.ndarray: HSI and GT data.
    """
    try:
        data = h5py.File(file_path, 'r')
        hsi_data = np.array(data['HSI'])
        gt_data = np.array(data['GT'])
        print(f"Loaded file: {file_path}")
        print(f"HSI shape: {hsi_data.shape}, GT shape: {gt_data.shape}")
        return hsi_data, gt_data
    except Exception as e:
        print(f"Error loading .mat file: {e}")
        return None, None

def split_into_tiles(hsi_data, gt_data, tile_size):
    """
    Split hsi_data and gt_data into non-overlapping tiles of size (tile_size x tile_size).
    
    Handles both cases:
      - If hsi_data is in (C, H, W) format (expected: channels=270, H, W)
      - If hsi_data is in (H, W, C) format, it transposes it.
    
    Returns:
        list, list: Lists of HSI tiles (each with shape (C, tile_size, tile_size)) and GT tiles.
    """
    # Check if the first dimension is the number of channels
    if hsi_data.shape[0] == 270:
        # Assume hsi_data is already in (C, H, W) format.
        channels, height, width = hsi_data.shape
    else:
        # Assume hsi_data is in (H, W, C) format and transpose it.
        hsi_data = np.transpose(hsi_data, (2, 0, 1))
        channels, height, width = hsi_data.shape

    hsi_tiles = []
    gt_tiles = []
    for i in range(0, height, tile_size):
        for j in range(0, width, tile_size):
            hsi_tile = hsi_data[:, i:i+tile_size, j:j+tile_size]
            gt_tile = gt_data[i:i+tile_size, j:j+tile_size]
            # Only add full-sized tiles.
            if hsi_tile.shape[1] == tile_size and hsi_tile.shape[2] == tile_size:
                hsi_tiles.append(hsi_tile)
                gt_tiles.append(gt_tile)
    return hsi_tiles, gt_tiles

def process_multiple_hsi_files(folder_path, tile_size=32):
    """
    Process multiple .mat files in a folder to create a combined dataset.
    
    Returns:
        np.ndarray, np.ndarray: Combined HSI tiles and GT tiles.
    """
    combined_hsi_tiles = []
    combined_gt_tiles = []
    
    for file_name in os.listdir(folder_path):
        if file_name.endswith('.mat'):
            file_path = os.path.join(folder_path, file_name)
            hsi_data, gt_data = load_hdf5_mat_file(file_path)
            if hsi_data is None or gt_data is None:
                continue
            hsi_tiles, gt_tiles = split_into_tiles(hsi_data, gt_data, tile_size)
            combined_hsi_tiles.extend(hsi_tiles)
            combined_gt_tiles.extend(gt_tiles)
    
    # Stack the tiles into NumPy arrays.
    combined_hsi_tiles = np.stack(combined_hsi_tiles)
    combined_gt_tiles = np.stack(combined_gt_tiles)
    
    print(f"Total number of HSI tiles: {combined_hsi_tiles.shape[0]}")
    print(f"Total number of GT tiles: {combined_gt_tiles.shape[0]}")
    return combined_hsi_tiles, combined_gt_tiles

# Set your folder path and tile size.
folder_path = r'C:\Users\ChloeAtherton\Capstone\data'
tile_size = 32

# Load and tile the data.
data, gt = process_multiple_hsi_files(folder_path, tile_size)

# Remap ground truths:
# Set background (0) to 255, and subtract 1 from all non-background labels.
gt[gt == 0] = 255
gt[gt != 255] -= 1

# Scale HSI data to [0,1].
min_value = np.min(data)
max_value = np.max(data)
data = ScaleData(data, min_value, max_value)


Loaded file: C:\Users\ChloeAtherton\Capstone\data\NC12.mat
HSI shape: (270, 2884, 682), GT shape: (2884, 682)
Loaded file: C:\Users\ChloeAtherton\Capstone\data\NC13.mat
HSI shape: (270, 808, 1098), GT shape: (808, 1098)
Loaded file: C:\Users\ChloeAtherton\Capstone\data\NC16.mat
HSI shape: (270, 976, 1060), GT shape: (976, 1060)
Total number of HSI tiles: 3730
Total number of GT tiles: 3730


In [4]:
# %% Patchification (with overlap)
patch_size = 32  # You can modify this patch size if needed
stride = 8       # Adjust the stride as required

# If data is a 4D array (n_tiles, channels, height, width)
if data.ndim == 4:
    n_tiles, channels, h, w = data.shape
    # Calculate necessary padding for the height and width dimensions of each tile.
    pad_h = patch_size * ((h // patch_size) + (0 if h % patch_size == 0 else 1)) - h
    pad_w = patch_size * ((w // patch_size) + (0 if w % patch_size == 0 else 1)) - w
    # Pad only the height and width dimensions (axis 2 and 3)
    data = np.pad(data, ((0, 0), (0, 0), (0, pad_h), (0, pad_w)), mode='constant')
else:
    # For a 3D array (channels, height, width)
    h, w = data.shape[1], data.shape[2]
    pad_h = patch_size * ((h // patch_size) + (0 if h % patch_size == 0 else 1)) - h
    pad_w = patch_size * ((w // patch_size) + (0 if w % patch_size == 0 else 1)) - w
    data = np.pad(data, ((0, 0), (0, pad_h), (0, pad_w)), mode='constant')

# For ground truth, assume gt is a 3D array (n_tiles, height, width)
if gt.ndim == 3:
    h_gt, w_gt = gt.shape[1], gt.shape[2]
    pad_h_gt = patch_size * ((h_gt // patch_size) + (0 if h_gt % patch_size == 0 else 1)) - h_gt
    pad_w_gt = patch_size * ((w_gt // patch_size) + (0 if w_gt % patch_size == 0 else 1)) - w_gt
    gt = np.pad(gt, ((0, 0), (0, pad_h_gt), (0, pad_w_gt)), mode='constant', constant_values=255)
else:
    gt = np.pad(gt, ((0,0), (0, pad_h), (0, pad_w)), mode='constant', constant_values=255)

print("After padding, data shape:", data.shape)
print("After padding, gt shape:", gt.shape)


# Define a function to extract overlapping patches from a 3D image (channels, height, width)
def extract_overlapping_patches_3d(image, patch_size, stride):
    patches = []
    # image shape: (channels, height, width)
    c, h, w = image.shape
    for i in range(0, h - patch_size + 1, stride):
        for j in range(0, w - patch_size + 1, stride):
            patch = image[:, i:i+patch_size, j:j+patch_size]
            patches.append(patch)
    return np.array(patches)

# Define a function to extract overlapping patches from a 2D image (height, width)
def extract_overlapping_patches_2d(image, patch_size, stride):
    patches = []
    # image shape: (height, width)
    h, w = image.shape
    for i in range(0, h - patch_size + 1, stride):
        for j in range(0, w - patch_size + 1, stride):
            patch = image[i:i+patch_size, j:j+patch_size]
            patches.append(patch)
    return np.array(patches)

# A unified function that works on 4D data: iterate over tiles and extract patches.
def extract_overlapping_patches_4d(data, patch_size, stride):
    all_patches = []
    for tile in data:
        if tile.ndim == 3:
            tile_patches = extract_overlapping_patches_3d(tile, patch_size, stride)
        elif tile.ndim == 2:
            tile_patches = extract_overlapping_patches_2d(tile, patch_size, stride)
        else:
            raise ValueError("Tile must be 2D or 3D")
        all_patches.extend(tile_patches)
    return np.array(all_patches)

# Extract overlapping patches from data and gt
data_patches = extract_overlapping_patches_4d(data, patch_size, stride)
gt_patches = extract_overlapping_patches_4d(gt, patch_size, stride)

# Filter patches: select only those with at least 30% valid (labeled) pixels.
label_fraction = 0.3
# For gt_patches, each patch is 2D (patch_size, patch_size)
filt = np.sum(gt_patches < 100, axis=(1, 2))
valid_idx = filt > ((patch_size ** 2) * label_fraction)
data_patches = data_patches[valid_idx]
gt_patches = gt_patches[valid_idx]

print("Dimension of patchified data:", data_patches.shape)
print("Dimension of patchified GT:", gt_patches.shape)


After padding, data shape: (3730, 270, 32, 32)
After padding, gt shape: (3730, 32, 32)
Dimension of patchified data: (1124, 270, 32, 32)
Dimension of patchified GT: (1124, 32, 32)


In [5]:
# %% Train-Validation Split

# Split patches into training and validation sets (80-20 split)
train_idx, val_idx = train_test_split(np.arange(data_patches.shape[0]), test_size=0.2, random_state=42)
train_data = data_patches[train_idx]
train_gt = gt_patches[train_idx]
validation_data = data_patches[val_idx]
validation_gt = gt_patches[val_idx]

print("Train data shape:", train_data.shape)
print("Train GT shape:", train_gt.shape)
print("Validation data shape:", validation_data.shape)
print("Validation GT shape:", validation_gt.shape)


Train data shape: (899, 270, 32, 32)
Train GT shape: (899, 32, 32)
Validation data shape: (225, 270, 32, 32)
Validation GT shape: (225, 32, 32)


In [6]:
import numpy as np

# Assuming gt_patches is your NumPy array of ground truth patches
unique_labels = np.unique(gt_patches)
print("Unique labels in ground truth patches:", unique_labels)


Unique labels in ground truth patches: [  0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15 255]


In [7]:
# %% Prepare Data for PyTorch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
batch_size = 128

# Convert NumPy arrays to PyTorch tensors
train_data_tensor = torch.tensor(train_data, dtype=torch.float32).to(device)
train_gt_tensor = torch.tensor(train_gt, dtype=torch.int64).to(device)
validation_data_tensor = torch.tensor(validation_data, dtype=torch.float32).to(device)
validation_gt_tensor = torch.tensor(validation_gt, dtype=torch.int64).to(device)

# Create DataLoaders
train_dataset = TensorDataset(train_data_tensor, train_gt_tensor)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataset = TensorDataset(validation_data_tensor, validation_gt_tensor)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)


cuda


In [10]:
import torch
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

print(f"Is CUDA available: {torch.cuda.is_available()}")
print(f"PyTorch version: {torch.__version__}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
import numpy as np

unique_classes = np.unique(train_gt_tensor.detach().cpu().numpy())
print("Unique classes in masks:", unique_classes)
print("Number of classes:", len(unique_classes))



Is CUDA available: True
PyTorch version: 2.5.1+cu124
Using device: cuda
Unique classes in masks: [  0   1   2   3   4   5   6   7   8   9  10  11  12  13 255]
Number of classes: 15


In [9]:
# %% Model Definition
# Example: Adjust your model definition to output 16 classes.
class SimpleSegmentationModel(nn.Module):
    def __init__(self, in_channels, out_channels=16, init_features=32):
        super(SimpleSegmentationModel, self).__init__()
        features = init_features
        # Encoder
        self.encoder1 = self._block(in_channels, features)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.encoder2 = self._block(features, features * 2)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        # Bottleneck
        self.bottleneck = self._block(features * 2, features * 4)
        # Decoder
        self.upconv2 = nn.ConvTranspose2d(features * 4, features * 2, kernel_size=2, stride=2)
        self.decoder2 = self._block(features * 4, features * 2)
        self.upconv1 = nn.ConvTranspose2d(features * 2, features, kernel_size=2, stride=2)
        self.decoder1 = self._block(features * 2, features)
        # Final Convolution updated for 16 classes
        self.conv = nn.Conv2d(in_channels=features, out_channels=out_channels, kernel_size=1)

    def forward(self, x):
        enc1 = self.encoder1(x)
        enc2 = self.encoder2(self.pool1(enc1))
        bottleneck = self.bottleneck(self.pool2(enc2))
        dec2 = self.upconv2(bottleneck)
        dec2 = torch.cat((dec2, enc2), dim=1)
        dec2 = self.decoder2(dec2)
        dec1 = self.upconv1(dec2)
        dec1 = torch.cat((dec1, enc1), dim=1)
        dec1 = self.decoder1(dec1)
        return self.conv(dec1)

    def _block(self, in_channels, features):
        return nn.Sequential(
            nn.Conv2d(in_channels, features, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(features),
            nn.ReLU(inplace=True),
            nn.Conv2d(features, features, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(features),
            nn.ReLU(inplace=True)
        )

# Initialize the model
input_channels = train_data.shape[1]  # e.g., 270 channels if your data is hyperspectral
num_classes = 16  # TODO: Change if your dataset has a different number of classes

model = SimpleSegmentationModel(in_channels=input_channels, out_channels=num_classes).to(device)

# Optional: print a summary of the model
try:
    from torchinfo import summary
    print(summary(model, input_size=(1, input_channels, patch_size, patch_size)))
except ImportError:
    print(model)


Layer (type:depth-idx)                   Output Shape              Param #
SimpleSegmentationModel                  [1, 16, 32, 32]           --
├─Sequential: 1-1                        [1, 32, 32, 32]           --
│    └─Conv2d: 2-1                       [1, 32, 32, 32]           77,760
│    └─BatchNorm2d: 2-2                  [1, 32, 32, 32]           64
│    └─ReLU: 2-3                         [1, 32, 32, 32]           --
│    └─Conv2d: 2-4                       [1, 32, 32, 32]           9,216
│    └─BatchNorm2d: 2-5                  [1, 32, 32, 32]           64
│    └─ReLU: 2-6                         [1, 32, 32, 32]           --
├─MaxPool2d: 1-2                         [1, 32, 16, 16]           --
├─Sequential: 1-3                        [1, 64, 16, 16]           --
│    └─Conv2d: 2-7                       [1, 64, 16, 16]           18,432
│    └─BatchNorm2d: 2-8                  [1, 64, 16, 16]           128
│    └─ReLU: 2-9                         [1, 64, 16, 16]           --
│  

In [10]:
# %% Training Setup

criterion = nn.CrossEntropyLoss(ignore_index=255)
optimizer = optim.Adam(model.parameters(), weight_decay=1e-4)
scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)
nb_epoch = 100  # Adjust number of epochs as needed

best_val_loss = float('inf')


In [11]:
def calculate_accuracy(output, target, ignore_index=255):
    """
    Calculate pixel-wise accuracy.

    Args:
        output (torch.Tensor): Model output of shape [batch, num_classes, height, width].
        target (torch.Tensor): Ground truth labels of shape [batch, height, width].
        ignore_index (int): Index to ignore in the target (e.g., background).

    Returns:
        accuracy (float): Pixel-wise accuracy.
    """
    device = output.device  # Ensure all new tensors are on the same device
    preds = torch.argmax(output, dim=1)  # Shape: [batch, height, width]

    if ignore_index is not None:
        mask = (target != ignore_index).to(device)
    else:
        mask = torch.ones_like(target, dtype=torch.bool, device=device)

    # Count correct predictions (excluding ignore index)
    correct = (preds[mask] == target[mask]).sum().float()
    total = mask.sum().float()

    accuracy = correct / (total + 1e-6)  # Add epsilon to avoid division by zero
    return accuracy.item()


def calculate_iou(outputs, target, num_classes, ignore_index=255):
    device = outputs.device  # get device from outputs
    preds = torch.argmax(outputs, dim=1)  # shape: [batch, H, W]
    intersection = torch.zeros(num_classes, device=device)
    union = torch.zeros(num_classes, device=device)
    
    for cls in range(num_classes):
        # Create masks for the current class
        pred_mask = (preds == cls)
        target_mask = (target == cls)
        
        if ignore_index is not None:
            valid_mask = (target != ignore_index)
            pred_mask = pred_mask & valid_mask
            target_mask = target_mask & valid_mask
        
        # Sum up intersection and union
        intersection[cls] = (pred_mask & target_mask).sum().float()
        union[cls] = (pred_mask | target_mask).sum().float()
    
    iou = intersection / (union + 1e-6)
    return iou


In [12]:
sample_image, sample_label = train_dataset[0]
print("Unique labels in first sample:", torch.unique(sample_label))


Unique labels in first sample: tensor([  1, 255], device='cuda:0')


In [15]:
import torch
from time import time

def train_and_validate_model(model, train_loader, val_loader, criterion, optimizer, scheduler, nb_epoch, nb_classes, device):
    """
    Trains and validates the model for a given number of epochs.
    
    Args:
        model (torch.nn.Module): The segmentation model.
        train_loader (DataLoader): DataLoader for training data.
        val_loader (DataLoader): DataLoader for validation data.
        criterion (torch.nn.Module): Loss function.
        optimizer (torch.optim.Optimizer): Optimizer.
        scheduler (torch.optim.lr_scheduler): Learning rate scheduler.
        nb_epoch (int): Number of epochs.
        nb_classes (int): Number of segmentation classes.
        device (torch.device): Device to use (cpu or cuda).
        
    Returns:
        list, list: Lists of average training and validation losses per epoch.
    """
    train_losses = []
    val_losses = []
    best_val_loss = float('inf')
    
    for epoch in range(nb_epoch):
        start_time = time()
        model.train()
        running_train_loss = 0.0
        for images, labels in train_loader:
            print("Unique labels:", torch.unique(labels))
            break
        # Training loop
        for images, labels in train_loader:

            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels.long())
            loss.backward()
            optimizer.step()
            running_train_loss += loss.item()
        scheduler.step()
        
        avg_train_loss = running_train_loss / len(train_loader)
        train_losses.append(avg_train_loss)
        
        # Validation loop
        model.eval()
        running_val_loss = 0.0
        total_accuracy = 0.0
        total_iou = torch.zeros(nb_classes).to(device)
        total_samples = 0
        
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels.long())
                running_val_loss += loss.item()
                
                # Calculate IoU and accuracy using your helper functions
                iou = calculate_iou(outputs, labels, nb_classes, ignore_index=255)
                accuracy = calculate_accuracy(outputs, labels, ignore_index=255)
                
                total_iou += iou
                total_accuracy += accuracy
                total_samples += 1
        
        avg_val_loss = running_val_loss / len(val_loader)
        val_losses.append(avg_val_loss)
        mean_iou = (total_iou / total_samples).mean().item()
        mean_accuracy = total_accuracy / total_samples
        
        epoch_time = time() - start_time
        print(f'Epoch [{epoch+1}/{nb_epoch}], Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}, Accuracy: {mean_accuracy:.4f}, IOU: {mean_iou:.4f}, Time: {epoch_time:.2f}s')
        
        # Save best model based on validation loss
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), 'best_model.pth')
    
    return train_losses, val_losses

# Example usage:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
train_losses, val_losses = train_and_validate_model(model, train_loader, val_loader, criterion, optimizer, scheduler, nb_epoch=200, nb_classes=15, device=device)


Unique labels: tensor([  0,   1,   2,   3,   4,   5,   7,   8,   9,  10,  11,  12,  13, 255],
       device='cuda:0')
Epoch [1/200], Train Loss: 0.5101, Val Loss: 0.3997, Accuracy: 0.8731, IOU: 0.5784, Time: 1.05s
Unique labels: tensor([  0,   1,   2,   3,   4,   5,   6,   7,   9,  10,  11,  12,  13, 255],
       device='cuda:0')
Epoch [2/200], Train Loss: 0.6060, Val Loss: 0.4014, Accuracy: 0.8747, IOU: 0.5834, Time: 0.29s
Unique labels: tensor([  0,   1,   2,   3,   4,   5,   7,   9,  10,  11,  12,  13, 255],
       device='cuda:0')
Epoch [3/200], Train Loss: 0.4520, Val Loss: 0.4247, Accuracy: 0.8576, IOU: 0.5563, Time: 0.28s
Unique labels: tensor([  0,   1,   2,   3,   4,   5,   7,   8,   9,  10,  11,  12,  13, 255],
       device='cuda:0')
Epoch [4/200], Train Loss: 0.4369, Val Loss: 0.4220, Accuracy: 0.8702, IOU: 0.5711, Time: 0.29s
Unique labels: tensor([  0,   1,   2,   3,   4,   5,   7,   8,   9,  10,  11,  12,  13, 255],
       device='cuda:0')
Epoch [5/200], Train Loss: 0.71

In [17]:
def evaluate_model(model, dataloader, criterion, nb_classes, device):
    """
    Evaluate the model on a dataset and compute loss, pixel accuracy, IoU, and Dice scores.

    Args:
        model (torch.nn.Module): The segmentation model.
        dataloader (DataLoader): DataLoader for the evaluation data.
        criterion (torch.nn.Module): Loss function.
        nb_classes (int): Number of segmentation classes.
        device (torch.device): Device (cpu or cuda).

    Returns:
        tuple: (pixel_accuracy, mean_iou, mean_dice)
    """
    model.eval()  # Set model to evaluation mode
    total_loss = 0.0
    total_correct = 0
    total_pixels = 0
    iou_scores = []
    dice_scores = []

    with torch.no_grad():
        for images, masks in dataloader:
            images, masks = images.to(device), masks.to(device)

            outputs = model(images)  # shape: (batch_size, num_classes, height, width)
            preds = torch.argmax(outputs, dim=1)  # predicted class for each pixel

            # Compute loss
            loss = criterion(outputs, masks)
            total_loss += loss.item()

            # Compute pixel-wise accuracy
            total_correct += (preds == masks).sum().item()
            total_pixels += masks.numel()

            # Compute IoU and Dice for each class
            for class_idx in range(nb_classes):
                pred_mask = (preds == class_idx)
                true_mask = (masks == class_idx)

                intersection = (pred_mask & true_mask).sum().item()
                union = (pred_mask | true_mask).sum().item()
                dice = 2 * intersection / (pred_mask.sum().item() + true_mask.sum().item() + 1e-6)

                if union > 0:
                    iou_scores.append(intersection / union)
                dice_scores.append(dice)

    pixel_accuracy = total_correct / total_pixels
    mean_iou = sum(iou_scores) / len(iou_scores) if len(iou_scores) > 0 else 0.0
    mean_dice = sum(dice_scores) / len(dice_scores) if len(dice_scores) > 0 else 0.0

    print(f"Validation Loss: {total_loss / len(dataloader):.4f}")
    print(f"Pixel Accuracy: {pixel_accuracy:.4f}")
    print(f"Mean IoU: {mean_iou:.4f}")
    print(f"Mean Dice Score: {mean_dice:.4f}")

    return pixel_accuracy, mean_iou, mean_dice


In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Assume nb_classes is 16
pixel_acc, mean_iou, mean_dice = evaluate_model(model, val_loader, criterion, nb_classes=16, device=device)


Validation Loss: 0.2617
Pixel Accuracy: 0.6126
Mean IoU: 0.4157
Mean Dice Score: 0.4956
